In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model
from tqdm import tqdm

# ==========================================
# 1. 경로 및 하이퍼파라미터 설정
# ==========================================
csv_path = "/content/drive/MyDrive/new_고컴비/train_dataset.csv"
output_lora_dir = "/content/drive/MyDrive/new_고컴비/hyebin_final_lora"

BATCH_SIZE = 64
LEARNING_RATE = 5e-5
EPOCHS = 3
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📡 현재 코랩 훈련 장치: [{device}]")

# ==========================================
# 2. 8:2 완벽 데이터 분할
# ==========================================
print("📊 마스터 데이터셋 로드 후 8:2 분할 진행 중...")
total_df = pd.read_csv(csv_path)
train_df, val_df = train_test_split(total_df, test_size=0.2, random_state=42)
print(f"🏋️ 훈련 세트(Train): {len(train_df)}장 | 🧪 검증 세트(Valid): {len(val_df)}장")

# ==========================================
# 3. 데이터셋 로더 정의
# ==========================================
class FashionCLIPDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            normalized_path = row['image_path'].replace("\\", "/")
            file_name = os.path.basename(normalized_path)
            cat = "bottom" if "bottom" in normalized_path.lower() else ("dress" if "dress" in normalized_path.lower() else "top")
            img_p = f"./Crop_Data_Detailed/{cat}/{file_name}"
            image = Image.open(img_p).convert("RGB")
            inputs = self.processor(text=[row['caption']], images=image, return_tensors="pt", padding="max_length", max_length=77, truncation=True)
            return {"inputs": {key: val.squeeze(0) for key, val in inputs.items()}, "raw_caption": row['caption']}
        except:
            return self.__getitem__((idx + 1) % len(self.df))

# ==========================================
# 4. 모델 및 프로세서 초기화 (LoRA 이식)
# ==========================================
print("\n🔄 오리지널 CLIP 모델 및 프로세서 로딩 중...")
base_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

config = LoraConfig(r=16, lora_alpha=32, target_modules=["q_proj", "v_proj"], lora_dropout=0.05, bias="none")
model = get_peft_model(base_model, config).to(device)

train_loader = DataLoader(FashionCLIPDataset(train_df, processor), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(FashionCLIPDataset(val_df, processor), batch_size=BATCH_SIZE, shuffle=False)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

style_candidates = ["기타", "레트로", "로맨틱", "리조트", "매니시", "모던", "밀리터리", "섹시", "소피스트케이티드", "스트리트", "스포티", "아방가르드", "오리엔탈", "웨스턴", "젠더리스", "컨트리", "클래식", "키치", "톰보이", "펑크", "페미닌", "프레피", "히피", "힙합"]
text_inputs = processor(text=style_candidates, return_tensors="pt", padding=True).to(device)

def get_true_style(caption):
    try:
        for p in caption.split(","):
            if "스타일" in p: return p.split(":")[1].strip()
    except: pass
    return "기타"

# ==========================================
# 5. 🏋️ 실시간 진행바 탑재 분리 학습 스타트!
# ==========================================
print("\n🎬 8:2 분할 신규 학습 및 실시간 채점 시작!")
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    progress_bar = tqdm(train_loader, desc=f"🏋️ Epoch [{epoch+1}/{EPOCHS}] 학습 중")

    for step, batch in enumerate(progress_bar):
        inputs = {k: v.to(device) for k, v in batch["inputs"].items()}
        outputs = model(**inputs, return_loss=True)

        optimizer.zero_grad()
        outputs.loss.backward()
        optimizer.step()

        loss_val = outputs.loss.item()
        total_train_loss += loss_val

        if step % 10 == 0:
            progress_bar.set_postfix({"Loss": f"{loss_val:.4f}"})

    # 에폭 종료 후 실시간 교차 채점
    model.eval()
    total_val_loss, correct_style_count, total_val_count = 0, 0, 0
    with torch.no_grad():
        # 1. 텍스트 특징 추출 안전장치
        text_features_output = model.get_text_features(**text_inputs)
        if hasattr(text_features_output, "pooler_output"):
            text_features = text_features_output.pooler_output
        else:
            text_features = text_features_output
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        for batch in tqdm(val_loader, desc=f"🧪 Epoch [{epoch+1}/{EPOCHS}] 채점 중"):
            inputs = {k: v.to(device) for k, v in batch["inputs"].items()}
            outputs = model(**inputs, return_loss=True)
            total_val_loss += outputs.loss.item()

            # ✨ [에러 수정 완료] 이미지 특징 추출부에도 동일하게 알맹이 텐서(.pooler_output)를 꺼내도록 격리 조치!
            image_features_output = model.get_image_features(pixel_values=inputs["pixel_values"])
            if hasattr(image_features_output, "pooler_output"):
                image_features = image_features_output.pooler_output
            else:
                image_features = image_features_output
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            similarity = (image_features @ text_features.T) * 100
            predictions = torch.argmax(similarity, dim=-1).cpu().numpy()

            for idx, pred_idx in enumerate(predictions):
                if style_candidates[pred_idx] == get_true_style(batch["raw_caption"][idx]):
                    correct_style_count += 1
                total_val_count += 1

    print(f"\n🏆 [Epoch {epoch+1} 종료 성적표]")
    print(f" 📈 Train Loss: {total_train_loss/len(train_loader):.4f} | 🧪 Valid Loss: {total_val_loss/len(val_loader):.4f}")
    print(f" 🎯 Valid Accuracy: {(correct_style_count/total_val_count)*100:.2f}%")
    print("="*65 + "\n")

model.save_pretrained(output_lora_dir)
print("✨ [학습 완료] 신규 가중치가 구글 드라이브에 성공적으로 업데이트되었습니다!")

# ==========================================
# 6. 📊 Baseline vs 신규 가중치 모델 최종 평가지표 비교 (R@1, R@5)
# ==========================================
print("\n⏱️ 최종 보고서용 필수 평가지표(R@1, R@5) 추출 시작...")
baseline_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
baseline_model.eval()

def evaluate_retrieval(eval_model, desc_name):
    r1_correct, r5_correct, total = 0, 0, 0
    with torch.no_grad():
        text_features_output = eval_model.get_text_features(**text_inputs)
        if hasattr(text_features_output, "pooler_output"):
            text_features = text_features_output.pooler_output
        else:
            text_features = text_features_output
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        for batch in tqdm(val_loader, desc=f"📊 {desc_name} 최종 채점 중"):
            inputs = {k: v.to(device) for k, v in batch["inputs"].items()}

            # ✨ [에러 수정 완료] 최종 성능 비교 함수 내 이미지 특징 추출부에도 완벽하게 이식!
            image_features_output = eval_model.get_image_features(pixel_values=inputs["pixel_values"])
            if hasattr(image_features_output, "pooler_output"):
                image_features = image_features_output.pooler_output
            else:
                image_features = image_features_output
            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

            similarity = (image_features @ text_features.T) * 100
            top5_preds = torch.topk(similarity, k=5, dim=-1).indices.cpu().numpy()

            for idx, preds in enumerate(top5_preds):
                real_style = get_true_style(batch["raw_caption"][idx])
                real_idx = style_candidates.index(real_style) if real_style in style_candidates else -1

                if preds[0] == real_idx: r1_correct += 1
                if real_idx in preds: r5_correct += 1
                total += 1
    return (r1_correct/total)*100, (r5_correct/total)*100

print("\n📊 1. Baseline (오리지널 CLIP) 채점 중...")
base_r1, base_r5 = evaluate_retrieval(baseline_model, "Baseline")

print("\n📊 2. 신규 고도화 모델 (지금 구운 LoRA) 채점 중...")
lora_r1, lora_r5 = evaluate_retrieval(model, "신규 LoRA")

print("\n📊 🏆 [최종 보고서 제출용 정량 지표 비교 스코어보드] 🏆 📊")
print("=" * 75)
print(f" ▶ Baseline (오리지널) ->  R@1(Accuracy): {base_r1:.2f}%  |  R@5: {base_r5:.2f}%")
print(f" ▶ 고도화 모델 (신규)  ->  R@1(Accuracy): {lora_r1:.2f}%  |  R@5: {lora_r5:.2f}%")
print("=" * 75)
print(f" 🔥 최종 결론: 새 가중치로 훈련 후 R@1 기준 성능이 [{lora_r1 - base_r1:.2f}% p] 대폭 향상됨!")

📡 현재 코랩 훈련 장치: [cuda]
📊 마스터 데이터셋 로드 후 8:2 분할 진행 중...
🏋️ 훈련 세트(Train): 48156장 | 🧪 검증 세트(Valid): 12039장

🔄 오리지널 CLIP 모델 및 프로세서 로딩 중...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


🎬 8:2 분할 신규 학습 및 실시간 채점 시작!


🧪 Epoch [1/3] 채점 중: 100%|██████████| 189/189 [03:03<00:00,  1.03it/s]



🏆 [Epoch 1 종료 성적표]
 📈 Train Loss: 1.6109 | 🧪 Valid Loss: 3.3421
 🎯 Valid Accuracy: 20.84%



...


📊 1. Baseline (오리지널 CLIP) 채점 중...


📊 Baseline 최종 채점 중: 100%|██████████| 189/189 [01:46<00:00,  1.77it/s]



📊 2. 신규 고도화 모델 (지금 구운 LoRA) 채점 중...


📊 신규 LoRA 최종 채점 중: 100%|██████████| 189/189 [01:49<00:00,  1.73it/s]


📊 🏆 [최종 보고서 제출용 정량 지표 비교 스코어보드] 🏆 📊
 ▶ Baseline (오리지널) ->  R@1(Accuracy): 0.32%  |  R@5: 8.54%
 ▶ 고도화 모델 (신규)  ->  R@1(Accuracy): 13.75%  |  R@5: 42.15%
 🔥 최종 결론: 새 가중치로 훈련 후 R@1 기준 성능이 [13.42% p] 대폭 향상됨!


In [ ]:
import os
print("📂 현재 위치:", os.getcwd())
print("📁 내 드라이브 안의 실제 폴더 목록:", os.listdir('.'))

📂 현재 위치: /content/drive/MyDrive/new_고컴비
📁 내 드라이브 안의 실제 폴더 목록: ['train_dataset.csv', 'Crop_Data_Detailed.zip', 'hyebin_final_lora', 'rola_find', 'FAISS', '8:2lorafinedtuning.ipynb']


In [ ]:
import os
import zipfile
import torch
import numpy as np
import pandas as pd
from transformers import CLIPModel, CLIPProcessor
from peft import PeftModel
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# ==========================================
# 🚨 [1단계] 날아간 이미지 압축 고속 해제 및 경로 복구
# ==========================================
# 세션 리셋으로 유실된 이미지 압축 파일을 코랩 로컬 공간(/content/)에 초고속으로 풉니다.
zip_path = "/content/drive/MyDrive/new_고컴비/Crop_Data_Detailed.zip"
extract_path = "/content/Crop_Data_Detailed_Extracted"

if not os.path.exists(extract_path):
    print("📦 코랩 가상환경 리셋으로 인해 이미지 압축 해제를 재진행합니다 (약 30초 소요)...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("✨ 압축 해제 완료!")
else:
    print("✅ 이미 압축이 풀려있습니다. 바로 연산을 시작합니다.")

csv_path = "/content/drive/MyDrive/new_고컴비/train_dataset.csv"
output_lora_dir = "/content/drive/MyDrive/new_고컴비/hyebin_final_lora"
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. 8:2 데이터 분할 (기존과 완전히 동일한 Valid 셋 확보)
# ==========================================
total_df = pd.read_csv(csv_path)
from sklearn.model_selection import train_test_split
_, val_df = train_test_split(total_df, test_size=0.2, random_state=42)

# 3. 프로세서 및 모델 로드
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
base_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)

# 혜빈이의 LoRA 가중치 결합
model = PeftModel.from_pretrained(base_model, output_lora_dir).to(device)
model.eval()

# ==========================================
# 4. 데이터 로더 세팅 (압축 풀린 로컬 경로 매핑)
# ==========================================
class FashionEvalDataset(Dataset):
    def __init__(self, dataframe, processor):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            normalized_path = row['image_path'].replace("\\", "/")
            file_name = os.path.basename(normalized_path)
            cat = "bottom" if "bottom" in normalized_path.lower() else ("dress" if "dress" in normalized_path.lower() else "top")

            # 방금 새로 푼 고속 로컬 경로로 정밀 타겟팅합니다!
            img_p = f"{extract_path}/Crop_Data_Detailed/{cat}/{file_name}"
            image = Image.open(img_p).convert("RGB")
            inputs = self.processor(images=image, return_tensors="pt")
            return {"pixel_values": inputs["pixel_values"].squeeze(0), "raw_caption": row['caption']}
        except:
            return self.__getitem__((idx + 1) % len(self.df))

val_loader = DataLoader(FashionEvalDataset(val_df, processor), batch_size=64, shuffle=False)

# 5. 24대 스타일 후보군 및 프롬프트 앙상블 템플릿 정의
style_candidates = ["기타", "레트로", "로맨틱", "리조트", "매니시", "모던", "밀리터리", "섹시", "소피스트케이티드", "스트리트", "스포티", "아방가르드", "오리엔탈", "웨스턴", "젠더리스", "컨트리", "클래식", "키치", "톰보이", "펑크", "페미닌", "프레피", "히피", "힙합"]

prompt_templates = [
    "A photo of a {} style clothing.",
    "An outfit with a {} mood.",
    "A close-up look of {} fashion item."
]

def get_true_style(caption):
    try:
        for p in caption.split(","):
            if "스타일" in p: return p.split(":")[1].strip()
    except: pass
    return "기타"

# 6. 텍스트 앙상블 특징 벡터 미리 연산
print("\n🔮 프롬프트 앙상블 텍스트 벡터 빌드 중...")
with torch.no_grad():
    ensemble_features = []
    for style in style_candidates:
        prompts = [template.format(style) for template in prompt_templates]
        txt_inputs = processor(text=prompts, return_tensors="pt", padding=True).to(device)
        txt_outputs = model.get_text_features(**txt_inputs)
        if hasattr(txt_outputs, "pooler_output"):
            txt_outputs = txt_outputs.pooler_output
        txt_outputs /= txt_outputs.norm(dim=-1, keepdim=True)
        mean_feature = txt_outputs.mean(dim=0)
        mean_feature /= mean_feature.norm(dim=-1)
        ensemble_features.append(mean_feature)
    text_features = torch.stack(ensemble_features)

# 7. ✨ 프롬프트 앙상블 기반 최종 성능 재채점 스타트!
print("📊 앙상블 기법 적용하여 12,039장 고속 재채점 진행 중...")
r1_correct, r5_correct, total = 0, 0, 0

with torch.no_grad():
    for batch in tqdm(val_loader, desc="지표 복구 연산 중"):
        pixel_values = batch["pixel_values"].to(device)

        img_outputs = model.get_image_features(pixel_values=pixel_values)
        if hasattr(img_outputs, "pooler_output"):
            image_features = img_outputs.pooler_output
        else:
            image_features = img_outputs
        image_features /= image_features.norm(dim=-1, keepdim=True)

        similarity = (image_features @ text_features.T) * 100
        top5_preds = torch.topk(similarity, k=5, dim=-1).indices.cpu().numpy()

        for idx, preds in enumerate(top5_preds):
            real_style = get_true_style(batch["raw_caption"][idx])
            real_idx = style_candidates.index(real_style) if real_style in style_candidates else -1

            if preds[0] == real_idx: r1_correct += 1
            if real_idx in preds: r5_correct += 1
            total += 1

print("\n📊 🏆 [프롬프트 앙상블 적용 최종 고도화 스코어보드] 🏆 📊")
print("=" * 75)
print(f" ▶ 최종 고도화 모델 ->  R@1(Accuracy): {(r1_correct/total)*100:.2f}%  |  R@5: {(r5_correct/total)*100:.2f}%")
print("=" * 75)

📦 코랩 가상환경 리셋으로 인해 이미지 압축 해제를 재진행합니다 (약 30초 소요)...
✨ 압축 해제 완료!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]


🔮 프롬프트 앙상블 텍스트 벡터 빌드 중...
📊 앙상블 기법 적용하여 12,039장 고속 재채점 진행 중...


지표 복구 연산 중: 100%|██████████| 189/189 [02:02<00:00,  1.54it/s]


📊 🏆 [프롬프트 앙상블 적용 최종 고도화 스코어보드] 🏆 📊
 ▶ 최종 고도화 모델 ->  R@1(Accuracy): 4.29%  |  R@5: 22.95%
